# Exploring the sources and designing Bronze and Silver

Plan of this notebook:

1. Look at each source (MITMA, INE, BOE) step by step, without assuming anything.
2. Decide what goes in Bronze and why.
3. Design Silver. We did not get it right the first time, so we keep the attempts that failed and what we learned from them. The hardest part is joining INE and MITMA by zone.

We use one sample week (6 to 12 March 2023) and plain DuckDB, like in class. To go faster, the trips are only loaded for one day (Monday 6 March).

In [1]:
import duckdb
import pandas as pd
import xml.etree.ElementTree as ET

RAW = '../data/raw'
MITMA = f'{RAW}/mitma/estudios_basicos'
ZON = f'{RAW}/mitma/zonificacion'
DAY = '20230306'

con = duckdb.connect()
con.sql("""
        INSTALL spatial; LOAD spatial; SET geometry_always_xy = true
    """)

SuccessBOOLEAN


---
## Part 1. Exploring the data

### 1.1 MITMA: trips

There is one compressed CSV per day and per zone level. We start with the districts file (with `all_varchar=true` option).

In [2]:
trips_file = f'{MITMA}/por-distritos/viajes/ficheros-diarios/2023-03/{DAY}_Viajes_distritos.csv.gz'

con.sql(f"""
        CREATE OR REPLACE TABLE trips_day AS SELECT * FROM read_csv('{trips_file}', delim='|', header=true, all_varchar=true)
    """)

CountBIGINT
18267798


In [3]:
con.sql("""
        SELECT * FROM trips_day LIMIT 10
    """)

fechaVARCHAR,periodoVARCHAR,origenVARCHAR,destinoVARCHAR,distanciaVARCHAR,actividad_origenVARCHAR,actividad_destinoVARCHAR,estudio_origen_posibleVARCHAR,estudio_destino_posibleVARCHAR,residenciaVARCHAR,rentaVARCHAR,edadVARCHAR,sexoVARCHAR,viajesVARCHAR,viajes_kmVARCHAR
20230306,19,01009_AM,01001,0.5-2,frecuente,casa,no,no,01,10-15,NA,NA,6.186,5.89
20230306,21,01009_AM,01001,0.5-2,frecuente,casa,no,no,01,10-15,NA,NA,2.587,2.486
20230306,14,01009_AM,01001,10-50,frecuente,casa,no,no,01,10-15,NA,NA,5.876,73.62
20230306,18,01017_AM,01001,10-50,frecuente,casa,no,no,01,10-15,NA,NA,2.587,33.682
20230306,20,01017_AM,01001,10-50,frecuente,casa,no,no,01,10-15,NA,NA,2.587,27.928
20230306,06,01051,01001,10-50,frecuente,casa,no,no,01,10-15,NA,NA,6.268,63.04
20230306,08,01051,01001,10-50,frecuente,casa,no,no,01,10-15,NA,NA,3.134,32.813
20230306,10,01051,01001,10-50,frecuente,casa,no,no,01,10-15,NA,NA,2.587,26.645
20230306,14,01051,01001,10-50,frecuente,casa,no,no,01,10-15,NA,NA,2.897,29.69
20230306,17,01051,01001,10-50,frecuente,casa,no,no,01,10-15,NA,NA,2.221,27.848


In [4]:
con.sql("""
        SELECT column_name, min, max, approx_unique, null_percentage 
        FROM (SUMMARIZE trips_day)
    """).df()

,column_name,min,max,approx_unique,null_percentage
0,fecha,20230306,20230306,1,0.0
1,periodo,00,23,24,0.0
2,origen,01001,externo,3902,0.0
3,destino,01001,externo,3902,0.0
4,distancia,0.5-2,>50,4,0.0
5,actividad_origen,casa,trabajo_estudio,4,0.0
6,actividad_destino,casa,trabajo_estudio,4,0.0
7,estudio_origen_posible,no,si,2,0.0
8,estudio_destino_posible,no,si,2,0.0
9,residencia,01,52,53,0.0


No column has real nulls: missing values come as the text `NA`, as we see below. Careful with `min` and `max` here, because everything is text and they are in alphabetical order.

In [5]:
con.sql("""
        SELECT COUNT(*) AS n_rows FROM trips_day
    """)

n_rowsBIGINT
18267798


18 million rows for one day and one level. Which values does each categorical column have?

In [6]:
for col in ['periodo', 'distancia', 'actividad_origen', 'actividad_destino', 'estudio_origen_posible', 'estudio_destino_posible', 'renta', 'edad', 'sexo']:
    print(col, con.sql(f"""
            SELECT LIST(DISTINCT {col} ORDER BY {col}) FROM trips_day
        """).fetchall()[0][0])

periodo ['00', '01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23']
distancia ['0.5-2', '10-50', '2-10', '>50']
actividad_origen ['casa', 'frecuente', 'no_frecuente', 'trabajo_estudio']
actividad_destino ['casa', 'frecuente', 'no_frecuente', 'trabajo_estudio']
estudio_origen_posible ['no', 'si']
estudio_destino_posible ['no', 'si']


renta ['10-15', '<10', '>15']
edad ['0-25', '25-45', '45-65', '65-100', 'NA']
sexo ['NA', 'hombre', 'mujer']


`edad` and `sexo` have the text `NA`. The MITMA dictionary (`LEEME_Especificaciones...xlsx`) says it is used when the value is not given "por criterios de privacidad". The dictionary also says the age bands are `0-24, 25-44, 45-64, >65`, but the file has `0-25, 25-45, 45-65, 65-100`. We trust the data and write down the difference.

How many trips are affected by these `NA`?

In [7]:
con.sql("""
        SELECT ROUND(100 * SUM(viajes::DOUBLE) FILTER (edad = 'NA') / SUM(viajes::DOUBLE), 1) AS pct_no_age,
               ROUND(100 * SUM(viajes::DOUBLE) FILTER (sexo = 'NA') / SUM(viajes::DOUBLE), 1) AS pct_no_sex
        FROM trips_day
    """)

pct_no_ageDOUBLE,pct_no_sexDOUBLE
24.0,37.3


So they are not rare: about a quarter of the trips has no age, and more than a third has no sex. We cannot delete those rows, we would lose real trips.

Next question: is there one row per combination of all the descriptive columns? If not, we would need to sum before using the data.

In [8]:
con.sql("""
        SELECT COUNT(*) AS repeated_combinations FROM (
            SELECT fecha, periodo, origen, destino, distancia, actividad_origen, actividad_destino,
                   estudio_origen_posible, estudio_destino_posible, residencia, renta, edad, sexo
            FROM trips_day GROUP BY ALL HAVING COUNT(*) > 1
        )
    """)

repeated_combinationsBIGINT
0


No repeated combinations, good. What about the numeric columns? `viajes` has decimals because MITMA expands the mobile sample to the population.

In [9]:
con.sql("""
        SELECT distancia, ROUND(SUM(viajes::DOUBLE)) AS trips, ROUND(SUM(viajes_km::DOUBLE) / SUM(viajes::DOUBLE), 1) AS avg_km,
               MIN(viajes::DOUBLE) AS min_trips, MAX(viajes::DOUBLE) AS max_trips
        FROM trips_day GROUP BY 1 ORDER BY avg_km
    """)

distanciaVARCHAR,tripsDOUBLE,avg_kmDOUBLE,min_tripsDOUBLE,max_tripsDOUBLE
0.5-2,52168716.0,1.2,1.0,1170.607
2-10,59094610.0,4.7,1.0,570.7
10-50,21699717.0,19.4,1.0,336.378
>50,2631957.0,151.9,1.0,99.0


The average km falls inside each band, so `viajes_km` looks like total km of those trips (not km per trip). 

Do the three levels contain the same trips? They should if they are the same data with a different zoning.

In [10]:
con.sql(f"""
        SELECT parse_filename(filename) AS file, COUNT(*) AS n_rows, ROUND(SUM(viajes::DOUBLE)) AS trips
        FROM read_csv('{MITMA}/por-*/viajes/ficheros-diarios/2023-03/{DAY}_Viajes_*.csv.gz', delim='|', header=true, all_varchar=true, filename=true)
        GROUP BY 1 ORDER BY 2 DESC
    """)

fileVARCHAR,n_rowsBIGINT,tripsDOUBLE
20230306_Viajes_distritos.csv.gz,18267798,135595000.0
20230306_Viajes_municipios.csv.gz,11254679,135595000.0
20230306_Viajes_GAU.csv.gz,6661956,135595000.0


Same total in the three, and fewer rows when zones get bigger. Now the whole week at district level, to see if weekends look different.

In [11]:
con.sql(f"""
        SELECT strptime(fecha, '%Y%m%d')::DATE AS day, dayname(strptime(fecha, '%Y%m%d')) AS day_name, ROUND(SUM(viajes::DOUBLE)) AS trips
        FROM read_csv('{MITMA}/por-distritos/viajes/ficheros-diarios/2023-03/*.csv.gz', delim='|', header=true, all_varchar=true)
        GROUP BY ALL ORDER BY day
    """)

dayDATE,day_nameVARCHAR,tripsDOUBLE
2023-03-06,Monday,135595000.0
2023-03-07,Tuesday,138939737.0
2023-03-08,Wednesday,138413033.0
2023-03-09,Thursday,139774895.0
2023-03-10,Friday,144299547.0
2023-03-11,Saturday,121659003.0
2023-03-12,Sunday,109006648.0


Weekdays are between 135 and 144 million trips, with Friday the highest, and Saturday and Sunday are clearly lower. So day types should show up in BQ1.

Now the zone codes. We print the origins that have a strange shape, meaning they are not numeric at the beginning.

In [12]:
con.sql("""
        SELECT CASE WHEN origen = 'externo' THEN 'externo'
                    WHEN origen LIKE 'FR%' THEN 'France'
                    WHEN origen LIKE 'PT%' THEN 'Portugal'
                    WHEN ends_with(origen, '_AM') THEN 'ends in _AM'
                    WHEN ends_with(origen, '_AD') THEN 'ends in _AD'
                    WHEN length(origen) = 5 THEN '5 digits'
                    WHEN length(origen) = 7 THEN '7 digits'
                    ELSE 'other' END AS kind,
               COUNT(DISTINCT origen) AS zones, MIN(origen) AS example, ROUND(SUM(viajes::DOUBLE)) AS trips
        FROM trips_day GROUP BY 1 ORDER BY zones desc
    """)

kindVARCHAR,zonesBIGINT,exampleVARCHAR,tripsDOUBLE
7 digits,1564,0105901,90060303.0
5 digits,1223,01001,33735306.0
ends in _AM,915,01004_AM,10430578.0
France,91,FR101,92906.0
ends in _AD,31,0306602_AD,1198733.0
Portugal,23,PT111,66529.0
externo,1,externo,10644.0


There are four kinds of Spanish code plus French and Portuguese regions and `externo`. So the zone ID must be text, and we will have to find out what each kind means.

How many trips stay inside the same zone, and how many touch a foreign zone?

In [13]:
con.sql("""
        SELECT ROUND(100 * SUM(viajes::DOUBLE) FILTER (origen = destino) / SUM(viajes::DOUBLE), 1) AS pct_same_zone,
               ROUND(100 * SUM(viajes::DOUBLE) FILTER (origen = 'externo' OR destino = 'externo' OR origen LIKE 'FR%' OR destino LIKE 'FR%' OR origen LIKE 'PT%' OR destino LIKE 'PT%') / SUM(viajes::DOUBLE), 2) AS pct_foreign
        FROM trips_day
    """)

pct_same_zoneDOUBLE,pct_foreignDOUBLE
26.5,0.22


About a quarter stay inside the zone. That matters later because the gravity model divides by the distance squared, and the distance inside a zone would be zero. Foreign trips are very few.

## 1.2 MITMA: overnight stays

In [14]:
nights_file = f'{MITMA}/por-distritos/pernoctaciones/ficheros-diarios/2023-03/{DAY}_Pernoctaciones_distritos.csv.gz'
con.sql(f"""
        SELECT * FROM read_csv('{nights_file}', delim='|', header=true, all_varchar=true) LIMIT 5
    """)

fechaVARCHAR,zona_residenciaVARCHAR,zona_pernoctacionVARCHAR,personasVARCHAR
20230306,01001,01001,2715.896
20230306,01001,01051,9.402
20230306,01001,01054_AM,3.289
20230306,01001,0105901,2.587
20230306,01001,0105902,30.730


In [15]:
con.sql(f"""
        SELECT column_name, min, max, approx_unique, null_percentage
        FROM (SUMMARIZE SELECT * FROM read_csv('{nights_file}', delim='|', header=true, all_varchar=true))
    """).df()

,column_name,min,max,approx_unique,null_percentage
0,fecha,20230306,20230306,1,0.0
1,zona_residencia,01001,5200108,3822,0.0
2,zona_pernoctacion,01001,5200108,3820,0.0
3,personas,0.000,9996.463,37007,0.0


We assumed that in the municipalities file both zones would be municipalities. Let's check it with the list of zone names of each level.

In [16]:
con.sql(f"""
        CREATE TEMP TABLE zone_names AS
        SELECT ID, name, regexp_extract(filename, 'zonificacion_([a-zA-Z]+)/', 1) AS zone_level
        FROM read_csv('{ZON}/zonificacion_*/nombres_*.csv', delim='|', header=true, all_varchar=true, union_by_name=true, filename=true)
    """)

con.sql(f"""
        WITH nights AS (
            SELECT zona_residencia, zona_pernoctacion, personas, regexp_extract(filename, 'por-([a-zA-Z]+)/', 1) AS zone_level
            FROM read_csv('{MITMA}/por-*/pernoctaciones/ficheros-diarios/2023-03/{DAY}_Pernoctaciones_*.csv.gz', delim='|', header=true, all_varchar=true, filename=true)
        )
        SELECT nights.zone_level, COUNT(*) AS n_rows, ROUND(SUM(personas::DOUBLE)) AS people,
               COUNT(*) FILTER (zona_residencia NOT IN (SELECT ID FROM zone_names WHERE zone_level = 'distritos')) AS residence_not_a_district,
               COUNT(*) FILTER (n.ID IS NULL) AS overnight_not_in_level
        FROM nights
        LEFT JOIN zone_names n ON n.zone_level = nights.zone_level AND n.ID = zona_pernoctacion
        GROUP BY 1
    """)

zone_levelVARCHAR,n_rowsBIGINT,peopleDOUBLE,residence_not_a_districtBIGINT,overnight_not_in_levelBIGINT
GAU,204957,47307399.0,0,0
distritos,356574,47307399.0,0,0
municipios,272709,47307399.0,0,0


We were wrong. `residence_not_a_district` is 0 in the three files: **the residence is always a district**, and only the overnight zone changes level. This surprised us, because the dictionary says that in the municipalities and GAU files the residence uses those zonings. The data says otherwise: the residence codes are the same 3,743 districts in the three files and no residence is a GAU name, so we trust the data and write down the difference. This changes the keys in Silver.

## 1.3 MITMA: zoning files

**Names.** First the municipalities file, without telling DuckDB there is a header.

In [17]:
con.sql(f"""
        SELECT * FROM read_csv('{ZON}/zonificacion_municipios/nombres_municipios.csv') LIMIT 3
    """)

column0BIGINT,column1VARCHAR,column2VARCHAR
NULL,ID,name
0,01001,Alegría-Dulantzi
1,01002,Amurrio


The first column is an index with no name, so DuckDB thinks there is no header. If you count these rows you get one zone too many (the header row). With `header=true` the counts are correct.

In [18]:
con.sql("""
        SELECT zone_level, COUNT(*) AS zones FROM zone_names GROUP BY 1
    """)

zone_levelVARCHAR,zonesBIGINT
distritos,3792
GAU,2086
municipios,2618


In [19]:
con.sql(f"""
        SELECT * FROM read_csv('{ZON}/zonificacion_distritos/nombres_distritos.csv', delim='|', header=true, all_varchar=true)
        WHERE ID IN ('01001', '01004_AM', '0105901', '0306602_AD', '34153153078')
        ORDER BY length(ID), ID
    """)

IDVARCHAR,nameVARCHAR
01001,Alegría-Dulantzi
0105901,Vitoria-Gasteiz distrito 01
01004_AM,Artziniega agregacion de municipios
0306602_AD,Elda distrito 02 agregacion de distritos
34153153078,Bardenas Reales


Now the codes make sense:
- `01001`: a municipality that is a district by itself
- `01004_AM`: several small municipalities grouped together (*agregacion de municipios*)
- `0105901`: a district of a big municipality (municipality code + district number)
- `_AD`: several districts grouped together
- 11-digit codes starting with `34`, like the Bardenas Reales. These look strange.

**The strange zones.** `34` is the INE code of Palencia, but Bardenas Reales is not in Palencia. We come back to them with the relation file.

**Geometries.** Shapefiles with polygons and centroids.

In [20]:
con.sql(f"""
        SELECT 'distritos' AS zone_level, COUNT(*) AS zones, COUNT(*) FILTER (ID LIKE 'FR%' OR ID LIKE 'PT%') AS foreign_zones FROM st_read('{ZON}/zonificacion_distritos/zonificacion_distritos_centroides.shp')
        UNION ALL
        SELECT 'municipios', COUNT(*), COUNT(*) FILTER (ID LIKE 'FR%' OR ID LIKE 'PT%') FROM st_read('{ZON}/zonificacion_municipios/zonificacion_municipios_centroides.shp')
        UNION ALL
        SELECT 'GAU', COUNT(*), COUNT(*) FILTER (ID LIKE 'FR%' OR ID LIKE 'PT%') FROM st_read('{ZON}/zonificacion_GAU/zonificacion_gaus_centroides.shp')
    """)

zone_levelVARCHAR,zonesBIGINT,foreign_zonesBIGINT
distritos,3909,117
municipios,2735,117
GAU,2203,117


In [21]:
con.sql(f"""
        SELECT OGC_FID, ID, st_astext(geom) AS geom, typeof(geom) AS geom_type
        FROM st_read('{ZON}/zonificacion_distritos/zonificacion_distritos_centroides.shp') LIMIT 2
    """)

OGC_FIDBIGINT,IDVARCHAR,geomVARCHAR,geom_typeVARCHAR
0,01001,POINT (539945.6938472011 4741948.88264596),GEOMETRY('EPSG:3042')
1,01002,POINT (502306.66342558316 4763642.870954381),GEOMETRY('EPSG:3042')


The coordinates are in metres (projected, EPSG:3042). There are 117 foreign zones with geometry that are not in the names files, and `OGC_FID` is only a row number added by the reader.

**The INE to MITMA relation.** This is the file we will need to join sources.

In [22]:
con.sql(f"""
        CREATE TEMP TABLE mapping_raw AS
        SELECT * FROM read_csv('{ZON}/relacion_ine_zonificacionMitma.csv', delim='|', header=true, all_varchar=true)
    """)

con.sql("""
        SELECT * FROM mapping_raw LIMIT 3
    """)

seccion_ineVARCHAR,distrito_ineVARCHAR,municipio_ineVARCHAR,distrito_mitmaVARCHAR,municipio_mitmaVARCHAR,gau_mitmaVARCHAR
0100101001,0100101,01001,01001,01001,01001
0100101002,0100101,01001,01001,01001,01001
0100201001,0100201,01002,01002,01002,01002


In [23]:
con.sql("""
        SELECT column_name, min, max, approx_unique, null_percentage FROM (SUMMARIZE mapping_raw)
    """).df()

,column_name,min,max,approx_unique,null_percentage
0,seccion_ine,0100101001,NA,36722,0.0
1,distrito_ine,0100101,NA,10001,0.0
2,municipio_ine,01001,NA,8055,0.0
3,distrito_mitma,01001,5200108,3890,0.0
4,municipio_mitma,01001,52001,2682,0.0
5,gau_mitma,01001,GAU Ávila,2150,0.0


In [24]:
con.sql("""
        SELECT COUNT(*) AS n_rows, COUNT(DISTINCT seccion_ine) AS sections, COUNT(*) FILTER (seccion_ine = 'NA') AS rows_with_NA,
               COUNT(DISTINCT distrito_mitma) AS districts, COUNT(DISTINCT municipio_mitma) AS municipalities, COUNT(DISTINCT gau_mitma) AS gaus
        FROM mapping_raw
    """)

n_rowsBIGINT,sectionsBIGINT,rows_with_NABIGINT,districtsBIGINT,municipalitiesBIGINT,gausBIGINT
36382,36334,49,3792,2618,2086


The 49 `NA` rows are the 49 strange zones from before: they are MITMA zones with no INE section, so nobody is registered there. They are unpopulated lands, not part of Palencia. The numbers of zones match the names files exactly. Before using it, we want to know how the levels fit inside each other. Does each smaller unit belong to only one bigger one?

In [25]:
con.sql("""
        SELECT 'district -> municipality' AS relation, MAX(n) AS max_zones, COUNT(*) FILTER (n > 1) AS with_more_than_one
        FROM (SELECT distrito_mitma, COUNT(DISTINCT municipio_mitma) AS n FROM mapping_raw WHERE seccion_ine <> 'NA' GROUP BY 1)
        UNION ALL
        SELECT 'municipality -> GAU', MAX(n), COUNT(*) FILTER (n > 1)
        FROM (SELECT municipio_mitma, COUNT(DISTINCT gau_mitma) AS n FROM mapping_raw WHERE seccion_ine <> 'NA' GROUP BY 1)
        UNION ALL
        SELECT 'section -> district', MAX(n), COUNT(*) FILTER (n > 1)
        FROM (SELECT seccion_ine, COUNT(DISTINCT distrito_mitma) AS n FROM mapping_raw WHERE seccion_ine <> 'NA' GROUP BY 1)
        UNION ALL
        SELECT 'INE municipality -> MITMA municipality', MAX(n), COUNT(*) FILTER (n > 1)
        FROM (SELECT municipio_ine, COUNT(DISTINCT municipio_mitma) AS n FROM mapping_raw WHERE seccion_ine <> 'NA' GROUP BY 1)
        UNION ALL
        SELECT 'INE district -> MITMA district', MAX(n), COUNT(*) FILTER (n > 1)
        FROM (SELECT distrito_ine, COUNT(DISTINCT distrito_mitma) AS n FROM mapping_raw WHERE seccion_ine <> 'NA' GROUP BY 1)
    """)

relationVARCHAR,max_zonesBIGINT,with_more_than_oneBIGINT
district -> municipality,1,0
municipality -> GAU,1,0
section -> district,1,0
INE municipality -> MITMA municipality,2,1
INE district -> MITMA district,2,5


Good: inside MITMA it is a clean hierarchy, and each section goes to one district. But there are some exceptions in the INE codes:

In [26]:
con.sql("""
        SELECT 'municipality' AS what, municipio_ine AS ine_code, LIST(DISTINCT municipio_mitma) AS mitma_zones FROM mapping_raw WHERE seccion_ine <> 'NA' GROUP BY 2 HAVING COUNT(DISTINCT municipio_mitma) > 1
        UNION ALL
        SELECT 'district', distrito_ine, LIST(DISTINCT distrito_mitma) FROM mapping_raw WHERE seccion_ine <> 'NA' GROUP BY 2 HAVING COUNT(DISTINCT distrito_mitma) > 1
    """)

whatVARCHAR,ine_codeVARCHAR,mitma_zonesVARCHAR[]
municipality,03903,"['03015', '03903']"
district,2408907,"['2408906', '2408907']"
district,2905101,"['2905101', '2905102']"
district,4718608,"['4718608', '4718607']"
district,0390301,"['0301502', '03903']"
district,2905102,"['2905102', '2905101']"


One INE municipality (03903, Los Montesinos) goes to two MITMA municipalities, and five INE districts go to two MITMA districts. We remember this for Part 3.

## 1.4 INE: household income atlas (ADRH)

We downloaded four tables. They have the same structure, except that the indicator column has a different name in each file.

In [27]:
con.sql(f"""
        CREATE TABLE adrh_raw AS SELECT * FROM read_csv('{RAW}/ine/adrh/*.csv', delim=';', header=true, all_varchar=true, union_by_name=true, filename=true)
    """)
con.sql("""
        SELECT Municipios, Distritos, Secciones, "Indicadores demográficos", Periodo, Total
        FROM adrh_raw WHERE "Indicadores demográficos" = 'Población' AND Periodo = '2023' AND Municipios LIKE '01001%'
    """)

MunicipiosVARCHAR,DistritosVARCHAR,SeccionesVARCHAR,Indicadores demográficosVARCHAR,PeriodoVARCHAR,TotalVARCHAR
01001 Alegría-Dulantzi,NULL,NULL,Población,2023,2.908
01001 Alegría-Dulantzi,0100101 Alegría-Dulantzi distrito 01,NULL,Población,2023,2.908
01001 Alegría-Dulantzi,0100101 Alegría-Dulantzi distrito 01,0100101001 Alegría-Dulantzi sección 01001,Población,2023,1.353
01001 Alegría-Dulantzi,0100101 Alegría-Dulantzi distrito 01,0100101002 Alegría-Dulantzi sección 01002,Población,2023,1.555


Important things we see here:
- Municipality, district and section rows are **mixed in the same table**. A municipality row has empty district and section. If we sum everything, we count the population three times.
- The code and the name are in the same cell (`01001 Alegría-Dulantzi`).
- `2.908` is two thousand and something, not 2.9. The dot is a thousands separator, and decimals use a comma. Let's see what a normal cast does:

In [28]:
con.sql("""
        SELECT '2.908'::DOUBLE AS normal_cast, REPLACE('2.908', '.', '')::DOUBLE AS correct, REPLACE(REPLACE('40,8', '.', ''), ',', '.')::DOUBLE AS with_comma
    """)

normal_castDOUBLE,correctDOUBLE,with_commaDOUBLE
2.908,2908.0,40.8


A normal cast would silently give us a population of 2.9 people, without any error. That is the most dangerous thing we have found so far. Now the rows and indicators:

In [29]:
con.sql("""
        SELECT parse_filename(filename) AS file,
               COALESCE("Indicadores demográficos", "Distribución por fuente de ingresos", "Indicadores de renta media") AS indicator,
               COUNT(*) AS rows_2023, COUNT(*) FILTER (Total IS NULL OR Total = '.') AS no_value
        FROM adrh_raw WHERE Periodo = '2023' GROUP BY ALL ORDER BY ALL
    """).df()

,file,indicator,rows_2023,no_value
0,indicadores_demograficos_30832.csv,Edad media de la población,55885,816
1,indicadores_demograficos_30832.csv,Población,55885,816
2,indicadores_demograficos_30832.csv,Porcentaje de hogares unipersonales,55885,816
3,indicadores_demograficos_30832.csv,Porcentaje de población de 65 y más años,55885,816
4,indicadores_demograficos_30832.csv,Porcentaje de población española,55885,816
5,indicadores_demograficos_30832.csv,Porcentaje de población menor de 18 años,55885,816
6,indicadores_demograficos_30832.csv,Tamaño medio del hogar,55885,816
7,renta_fuentes_ingreso_30825.csv,Fuente de ingreso: otras prestaciones,55885,5030
8,renta_fuentes_ingreso_30825.csv,Fuente de ingreso: otros ingresos,55885,5030
9,renta_fuentes_ingreso_30825.csv,Fuente de ingreso: pensiones,55885,5030


Some income indicators are missing much more often than population (around 5,000 rows against 816). We also noticed that the whole 2015 to 2024 series came with the download, and we only need 2023. Let's see the population by territorial level in 2023:

In [30]:
con.sql("""
        SELECT CASE WHEN Secciones IS NOT NULL THEN 'section' WHEN Distritos IS NOT NULL THEN 'district' ELSE 'municipality' END AS level,
               COUNT(*) AS n_rows, COUNT(TRY_CAST(REPLACE(Total, '.', '') AS double)) AS with_value, SUM(TRY_CAST(REPLACE(Total, '.', '') AS double)) AS population
        FROM adrh_raw WHERE "Indicadores demográficos" = 'Población' AND Periodo = '2023' GROUP BY 1
    """)

levelVARCHAR,n_rowsBIGINT,with_valueBIGINT,populationDOUBLE
section,37229,36477,47972964.0
municipality,8139,8132,47972964.0
district,10517,10460,47972964.0


The three levels add up to exactly the same population, and that's a good sign: the total of Spain in this table is 47,972,964. We will use this number to check our joins. But 752 section rows are empty. Why? Let's take one empty section and look at its history.

In [31]:
con.sql("""
        SELECT Secciones, Periodo, Total FROM adrh_raw
        WHERE "Indicadores demográficos" = 'Población' AND Secciones LIKE '0105901007%' ORDER BY Periodo
    """)

SeccionesVARCHAR,PeriodoVARCHAR,TotalVARCHAR
0105901007 Vitoria-Gasteiz sección 01007,2015,804
0105901007 Vitoria-Gasteiz sección 01007,2016,821
0105901007 Vitoria-Gasteiz sección 01007,2017,839
0105901007 Vitoria-Gasteiz sección 01007,2018,NULL
0105901007 Vitoria-Gasteiz sección 01007,2019,NULL
0105901007 Vitoria-Gasteiz sección 01007,2020,NULL
0105901007 Vitoria-Gasteiz sección 01007,2021,NULL
0105901007 Vitoria-Gasteiz sección 01007,2022,NULL
0105901007 Vitoria-Gasteiz sección 01007,2023,NULL
0105901007 Vitoria-Gasteiz sección 01007,2024,NULL


This section had people until 2017 and then nothing. It looks like a section that stopped existing and got renumbered, so an empty value doesn't always mean "secret" or "zero". We should still read the INE methodology to confirm the reasons for the nulls (the professor asked for this).

## 1.5 INE: businesses (DIRCE, table 4721)

In [32]:
con.sql(f"""
        CREATE TABLE dirce_raw AS SELECT * FROM read_csv('{RAW}/ine/dirce/empresas_municipio_4721.csv', delim=';', header=true, all_varchar=true, filename=true)
    """)
con.sql("""
        SELECT "Totales Territoriales", Provincias, Municipios, "Grupos CNAE", Total FROM dirce_raw WHERE Periodo = '2023' AND "Grupos CNAE" = 'Total' AND (Municipios LIKE '01001%' OR Provincias IS NULL OR (Municipios IS NULL AND Provincias LIKE '01%'))
    """)

Totales TerritorialesVARCHAR,ProvinciasVARCHAR,MunicipiosVARCHAR,Grupos CNAEVARCHAR,TotalVARCHAR
Total Nacional,NULL,NULL,Total,3.207.580
Total Nacional,01 Araba/Álava,NULL,Total,17.629
Total Nacional,01 Araba/Álava,01001 Alegría-Dulantzi,Total,150


Again national, province and municipality rows are mixed. How complete is it?

In [33]:
con.sql("""
        SELECT "Grupos CNAE" AS grp, COUNT(*) AS municipalities, COUNT(TRY_CAST(REPLACE(Total, '.', '') AS bigint)) AS with_value,
               SUM(TRY_CAST(REPLACE(Total, '.', '') AS bigint)) AS sum_municipalities,
               (SELECT REPLACE(n.Total, '.', '')::BIGINT FROM dirce_raw n WHERE n.Periodo = '2023' AND n.Provincias IS NULL AND n."Grupos CNAE" = d."Grupos CNAE") AS national
        FROM dirce_raw d WHERE Periodo = '2023' AND Municipios IS NOT NULL GROUP BY 1 ORDER BY with_value desc
    """)

grpVARCHAR,municipalitiesBIGINT,with_valueBIGINT,sum_municipalitiesHUGEINT,nationalBIGINT
Total,8136,6875,3205679,3207580
"Comercio, transporte y hostelería",8136,3141,1071915,1105590
Total servicios,8136,3120,1522044,1547129
F Construcción,8136,2996,360796,376996
Industria,8136,2971,168331,177865
MN Actividades profesionales y técnicas,8136,1313,559767,599322
Otros servicios personales,8136,1313,296293,329579
"Educación, sanidad y servicios sociales",8136,1313,258863,277554
L Actividades inmobiliarias,8136,1236,182669,195428
K Actividades financieras y de seguros,8136,1136,68386,75673


**Provinces and regions.** None of the files above says which region (CCAA) each province belongs to, and we will need it for the holidays. INE has this list on a web page (`cod_ccaa_provincia.htm`). It isn't a CSV, so we saved the table as `ine/ccaa_provincias.csv`, keeping the same columns.

In [34]:
con.sql(f"""
        CREATE TEMP TABLE provinces_raw AS
        SELECT * FROM read_csv('{RAW}/ine/ccaa_provincias.csv', delim=';', header=true, all_varchar=true, filename=true)
    """)

con.sql("""
        SELECT * EXCLUDE (filename) FROM provinces_raw LIMIT 3
    """)

CODAUTOVARCHAR,Comunidad AutónomaVARCHAR,CPROVARCHAR,ProvinciaVARCHAR
01,Andalucía,04,Almería
01,Andalucía,11,Cádiz
01,Andalucía,14,Córdoba


In [35]:
con.sql("""
        SELECT COUNT(*) AS n_rows, COUNT(DISTINCT CPRO) AS provinces, COUNT(DISTINCT CODAUTO) AS regions FROM provinces_raw
    """)

n_rowsBIGINT,provincesBIGINT,regionsBIGINT
52,52,19


The `Total` group exists in most municipalities and the sum is almost the national figure. The detailed groups are empty in most municipalities (for example only about 1,100 of 8,136 for financial activities), but the ones that exist seem to be the bigger towns, because the sums still reach 90 to 95% of the national figure.

## 1.6 BOE: public holidays 2023

It is an XML with a table inside. Rows are holidays, columns are regions.

In [36]:
root = ET.parse(f'{RAW}/boe/fiestas_laborales/2023_BOE-A-2022-16755.xml').getroot()
rows = [[' '.join(''.join(cell.itertext()).split()) for cell in tr] for tr in root.iter('tr')]

for row in rows[:10]:
    print(row[:10], '...', len(row), 'cells')

['Fecha de las fiestas', 'Comunidades Autónomas'] ... 2 cells
['Andalucía', 'Aragón', 'Asturias', 'Illes Balears', 'Canarias (1)', 'Cantabria', 'Castilla-La Mancha', 'Castilla y León', 'Cataluña (2)', 'Extremadura'] ... 19 cells
['ENERO', '', '', '', '', '', '', '', '', ''] ... 20 cells
['2 Lunes siguiente a Año Nuevo.', '**', '**', '**', '', '', '', '', '**', ''] ... 20 cells
['6 Epifanía del Señor.', '**', '**', '**', '**', '**', '**', '**', '**', '**'] ... 20 cells
['FEBRERO', '', '', '', '', '', '', '', '', ''] ... 20 cells
['21 Martes de Carnaval.', '', '', '', '', '', '', '', '', ''] ... 20 cells
['28 Día de Andalucía.', '***', '', '', '', '', '', '', '', ''] ... 20 cells
['MARZO', '', '', '', '', '', '', '', '', ''] ... 20 cells
['1 Día de les Illes Balears.', '', '', '', '***', '', '', '', '', ''] ... 20 cells


The first two rows are the header, and the rows with only a month are separators. To know what the stars mean, we read the text before the table:

In [37]:
for p in root.iter('p'):
    text = ' '.join(''.join(p.itertext()).split())
    if '(*' in text or text[:2] in ('1.', '2.', '3.'):
        print('-', text[:400])

- – Fiesta Nacional no sustituible (*).
- – Fiesta Nacional respecto de la que no se ha ejercido la facultad de sustitución (**).
- – Fiesta de Comunidad Autónoma (***).
- 1. En la Comunidad Autónoma de Canarias, el Decreto 156/2022, de 30 de junio, por el que se determina el calendario de fiestas laborales de la Comunidad Autónoma de Canarias para el año 2023, y se abre plazo para fijar las fiestas locales (BOC de 12 de julio de 2022) dispone que: «En las islas de El Hierro, Fuerteventura, Gran Canaria, La Gomera, La Palma, Lanzarote y Tenerife, las fiestas laboral
- 2. En la Comunidad Autónoma de Cataluña, la Orden EMT/81/2022, de 21 de abril, por la que se establece el calendario oficial de fiestas laborales en Cataluña para el año 2023 (DOGC de 28 de abril de 2022) dispone que: «En el territorio de Arán, de acuerdo con el Decreto 152/1997, de 25 de junio, y el Decreto 146/1998, de 23 de junio, que modifican el Decreto 177/1980, de 3 de octubre, sobre el cal
- 3. En la Ciudad Autóno

We have the meaning of `*`, `**` and `***`. The notes at the end add some local cases that are not in the table (Canary Islands, Arán and one extra day in Ceuta). They are too local for us, so we only use the table.

---
# Part 2. What goes in Bronze and why

The project says Bronze is "raw data without transformation, stored as-is for auditability". We still had to make some decisions.

### General decisions

| Decision | Why (what we saw in Part 1) |
|---|---|
| All columns as text | Letting DuckDB guess failed (`estudio_destino_posible`), and casting would be dangerous (`2.908` becomes 2.9). `NA` and `.` also break numeric columns. If a cast is wrong we want to fix it in Silver without downloading again. |
| Keep original column names and values | So that we can compare any row with the source file. `NA`, `.` and `01001 Alegría-Dulantzi` stay as they are. |
| One table per dataset with `zone_level`, not one per level | The columns are the same in the three levels, and the level is only in the folder name. Without this column, the same ID `01001` in different levels could not be told apart. |
| Add `_source_file` and `_ingested_at` | To know where a row comes from, and to load only the new days later. |
| Keep all years in the INE tables | It's what the source gives. Filtering to 2023 is a decision, so it goes to Silver. |
| Partition trips and overnight stays by `zone_level`, `fecha` | The project asks for it and nearly all queries will filter by level and by day. |

### Column by column

**`bronze_mitma_od_daily`**: all 15 source columns + `zone_level`. We thought about dropping `estudio_*_posible` because we don't need them in our questions, but Bronze shouldn't decide that.

**`bronze_mitma_overnight_daily`**: the 4 source columns + `zone_level`.

**`bronze_mitma_zone_names`**: `ID`, `name`, `zone_level`. Here we do remove something: the index column in the municipalities file, because it is not data and it doesn't exist in the other two files.

**`bronze_mitma_zone_geometries`**: `ID`, `geom`, plus `geometry_type` (polygon or centroid) and `zone_level`. `geom` is the only column that is not text, because the shapefile isn't text. We drop `OGC_FID` (a row number from the reader). We keep the foreign zones.

**`bronze_mitma_zone_mapping`**: the 6 source columns, including the `NA` rows.

**`bronze_ine_adrh`**: the 4 tables together, with the indicator column renamed to `indicador` (it has a different name in each file) and `table_id` to know the origin. We don't split the code from the name here.

**`bronze_ine_dirce`**: the 6 source columns as they are.

**`bronze_ine_provinces`**: the 4 columns of the INE list (`CODAUTO`, `Comunidad Autónoma`, `CPRO`, `Provincia`) as they are. It is the only file that links provinces and regions.

**`bronze_boe_holidays`**: `year`, `mes`, `fiesta`, `ccaa`, `marca`. `year` is added because the table only gives the day and the month, and the year is only in the file name. This is the only table that changes shape, because an XML table can't be stored as it is. We make one row per holiday and region, and the text is not touched (`Cataluña (2)`, `6 Epifanía del Señor.`).

Now we create them. For this test, trips are only one day of districts, and overnight stays one day of the three levels.

In [38]:
con.sql(f"""
        CREATE TABLE bronze_mitma_od_daily AS
        SELECT *, 'distritos' AS zone_level, '{trips_file}' AS _source_file, NOW() AS _ingested_at FROM trips_day
    """)
con.sql("""
        DROP TABLE trips_day
    """)

con.sql(f"""
        CREATE TABLE bronze_mitma_overnight_daily AS
        SELECT fecha, zona_residencia, zona_pernoctacion, personas, regexp_extract(filename, 'por-([a-zA-Z]+)/', 1) AS zone_level, filename AS _source_file, NOW() AS _ingested_at
        FROM read_csv('{MITMA}/por-*/pernoctaciones/ficheros-diarios/2023-03/{DAY}_Pernoctaciones_*.csv.gz', delim='|', header=true, all_varchar=true, filename=true)
    """)

con.sql(f"""
        CREATE TABLE bronze_mitma_zone_names AS
        SELECT ID, name, regexp_extract(filename, 'zonificacion_([a-zA-Z]+)/', 1) AS zone_level, filename AS _source_file, NOW() AS _ingested_at
        FROM read_csv('{ZON}/zonificacion_*/nombres_*.csv', delim='|', header=true, all_varchar=true, union_by_name=true, filename=true)
    """)

con.sql(f"""
        CREATE TABLE bronze_mitma_zone_mapping AS
        SELECT seccion_ine, distrito_ine, municipio_ine, distrito_mitma, municipio_mitma, gau_mitma, filename AS _source_file, NOW() AS _ingested_at
        FROM read_csv('{ZON}/relacion_ine_zonificacionMitma.csv', delim='|', header=true, all_varchar=true, filename=true)
    """)

CountBIGINT
36382


In [39]:
geoms = []
for level, name in [('distritos', 'distritos'), ('municipios', 'municipios'), ('GAU', 'gaus')]:
    for kind, suffix in [('polygon', ''), ('centroid', '_centroides')]:
        shp = f'{ZON}/zonificacion_{level}/zonificacion_{name}{suffix}.shp'
        geoms.append(f"SELECT ID, geom, '{kind}' AS geometry_type, '{level}' AS zone_level, '{shp}' AS _source_file, NOW() AS _ingested_at FROM st_read('{shp}')")

con.sql("CREATE TABLE bronze_mitma_zone_geometries AS " + " UNION ALL ".join(geoms))

CountBIGINT
17694


In [40]:
con.sql("""
        CREATE TABLE bronze_ine_adrh AS
        SELECT Municipios, Distritos, Secciones,
               COALESCE("Indicadores demográficos", "Distribución por fuente de ingresos", "Indicadores de renta media") AS indicador,
               Periodo, Total, regexp_extract(filename, '([0-9]+).csv', 1) AS table_id, filename AS _source_file, NOW() AS _ingested_at
        FROM adrh_raw
    """)
con.sql("""
        DROP TABLE adrh_raw
    """)

con.sql("""
        CREATE TABLE bronze_ine_dirce AS
        SELECT * EXCLUDE (filename), filename AS _source_file, NOW() AS _ingested_at FROM dirce_raw
    """)
con.sql("""
        DROP TABLE dirce_raw
    """)

con.sql("""
        CREATE TABLE bronze_ine_provinces AS
        SELECT CODAUTO, "Comunidad Autónoma", CPRO, Provincia, filename AS _source_file, NOW() AS _ingested_at FROM provinces_raw
    """)

CountBIGINT
52


In [41]:
# BOE: from the table to one row per holiday and region
regions = rows[1]
holidays = []
month = None
for row in rows[2:]:
    if all(cell == '' for cell in row[1:]):
        month = row[0]
        continue
    for region, mark in zip(regions, row[1:]):
        holidays.append((month, row[0], region, mark))

df_holidays = pd.DataFrame(holidays, columns=['mes', 'fiesta', 'ccaa', 'marca'])

con.sql(f"""
        CREATE TABLE bronze_boe_holidays AS
        SELECT regexp_extract('{RAW}/boe/fiestas_laborales/2023_BOE-A-2022-16755.xml', '/([0-9]+)_BOE', 1) AS year, *, '{RAW}/boe/fiestas_laborales/2023_BOE-A-2022-16755.xml' AS _source_file, NOW() AS _ingested_at FROM df_holidays
    """)

CountBIGINT
722


In [42]:
con.sql("""
        SELECT table_name, estimated_size AS approx_rows FROM duckdb_tables() WHERE table_name LIKE 'bronze%' ORDER BY 1
    """)

table_nameVARCHAR,approx_rowsBIGINT
bronze_boe_holidays,722
bronze_ine_adrh,11177000
bronze_ine_dirce,1261106
bronze_ine_provinces,52
bronze_mitma_od_daily,18267798
bronze_mitma_overnight_daily,834240
bronze_mitma_zone_geometries,17694
bronze_mitma_zone_mapping,36382
bronze_mitma_zone_names,8496


In [43]:
con.sql("""
        DESCRIBE bronze_ine_adrh
    """)

column_nameVARCHAR,column_typeVARCHAR,nullVARCHAR,keyVARCHAR,defaultVARCHAR,extraVARCHAR
Municipios,VARCHAR,YES,NULL,NULL,NULL
Distritos,VARCHAR,YES,NULL,NULL,NULL
Secciones,VARCHAR,YES,NULL,NULL,NULL
indicador,VARCHAR,YES,NULL,NULL,NULL
Periodo,VARCHAR,YES,NULL,NULL,NULL
Total,VARCHAR,YES,NULL,NULL,NULL
table_id,VARCHAR,YES,NULL,NULL,NULL
_source_file,VARCHAR,YES,NULL,NULL,NULL
_ingested_at,TIMESTAMP WITH TIME ZONE,YES,NULL,NULL,NULL


---
# Part 3. Designing Silver

### What do our questions need?

- **BQ1** typical day: trips per hour and day, and knowing which days are weekends or holidays.
- **BQ2** gravity model `T_ij = k * P_i * E_j / d_ij^2`: real trips between zones, population of the origin (`P_i`), economic activity of the destination (`E_j`) and the distance (`d_ij`).
- **BQ3** overnight stays: where people sleep against where they live.

### First idea: one big table

The project gives `silver_integrated_od` as an example, so the first idea was one table with every trip and the data of its origin and destination next to it. We discarded it quickly: a day of districts has 18 million rows and only about 3,900 zones, so we would copy the same zone information millions of times, and if a value changes we would have to rewrite everything. BQ1 doesn't even need population. So we go for a **star schema**, like in the Kimball model: facts (trips, overnight stays) in the centre and small dimensions around them. Population and economy per zone go to their own table.

That means that the key thing to define is what a zone is.

## 3.1 The zone dimension

**Attempt 1: one table per level** (`dim_district`, `dim_municipality`, `dim_gau`). Each one would have `zone_id` as a primary key. It looks clean, but the fact tables would need three versions of each foreign key or three fact tables, while the columns are all the same.

**Attempt 2: one table with all levels and `zone_id` as primary key.** Let's see if the ID identifies a zone by itself.

In [44]:
con.sql("""
        SELECT n_levels, COUNT(*) AS ids FROM (SELECT ID, COUNT(DISTINCT zone_level) AS n_levels FROM bronze_mitma_zone_names GROUP BY 1)
        GROUP BY 1 ORDER BY 1
    """)

n_levelsBIGINT,idsBIGINT
1,1915
2,574
3,1811


In [45]:
con.sql("""
        SELECT zone_level, ID, name FROM bronze_mitma_zone_names WHERE ID = '01001' OR ID = '08019' OR ID = '0801901' ORDER BY ID, zone_level
    """)

zone_levelVARCHAR,IDVARCHAR,nameVARCHAR
GAU,01001,Alegría-Dulantzi
distritos,01001,Alegría-Dulantzi
municipios,01001,Alegría-Dulantzi
municipios,08019,Barcelona
distritos,0801901,Barcelona distrito 01


It doesn't: many IDs exist in the three levels. Often it is the same place, but a level can also have a code that another level doesn't have. So **attempt 3: primary key (`zone_level`, `zone_id`)**. That's why every foreign key to this table needs both columns.

Which zones must be in the table? Not only the ones in the names files. The trips also use foreign zones. We check it on the three levels, because if a zone is missing, the join with trips loses rows.

In [46]:
con.sql(f"""
        CREATE TEMP TABLE zones_in_trips AS
        SELECT DISTINCT regexp_extract(filename, 'por-([a-zA-Z]+)/', 1) AS zone_level, origen AS ID
        FROM read_csv('{MITMA}/por-*/viajes/ficheros-diarios/2023-03/{DAY}_Viajes_*.csv.gz', delim='|', header=true, all_varchar=true, filename=true)
    """)

con.sql("""
        SELECT t.zone_level, COUNT(*) AS zones_in_trips, COUNT(*) FILTER (n.ID IS NULL) AS not_in_names,
               COUNT(*) FILTER (n.ID IS NULL AND g.ID IS NULL) AS not_in_names_nor_geometries
        FROM zones_in_trips t
        LEFT JOIN bronze_mitma_zone_names n ON t.zone_level = n.zone_level AND t.ID = n.ID
        LEFT JOIN (SELECT DISTINCT zone_level, ID FROM bronze_mitma_zone_geometries) g ON t.zone_level = g.zone_level AND t.ID = g.ID
        GROUP BY 1
    """)

zone_levelVARCHAR,zones_in_tripsBIGINT,not_in_namesBIGINT,not_in_names_nor_geometriesBIGINT
municipios,2675,115,1
GAU,2143,115,1
distritos,3848,115,1


In [47]:
con.sql("""
        SELECT n.zone_level, COUNT(*) AS zones_in_names_without_trips
        FROM bronze_mitma_zone_names n
        LEFT JOIN zones_in_trips t ON n.zone_level = t.zone_level AND n.ID = t.ID
        WHERE t.ID IS NULL
        GROUP BY 1
    """)

zone_levelVARCHAR,zones_in_names_without_tripsBIGINT
GAU,58
municipios,58
distritos,59


So:
- Foreign zones are in the geometries but not in the names. We take them from the geometries and use the code as the name.
- `externo` isn't anywhere, so we add it by hand, with no geometry.
- In districts, 59 zones have no trips that day: the 49 unpopulated lands and 10 others.

**The province.** First we wanted `left(zone_id, 2)`, but section 1.3 showed it puts 49 zones in Palencia. And the GAU IDs are names like `GAU Madrid`. But the relation file has the INE municipality of each section, and its first two digits are the province. Does a zone ever cover two provinces? If so, it would have no single province.

In [48]:
con.sql("""
        SELECT 'distritos' AS zone_level, MAX(n) AS max_provinces
        FROM (SELECT distrito_mitma, COUNT(DISTINCT LEFT(municipio_ine, 2)) AS n FROM bronze_mitma_zone_mapping WHERE seccion_ine <> 'NA' GROUP BY 1)
        UNION ALL
        SELECT 'municipios', MAX(n)
        FROM (SELECT municipio_mitma, COUNT(DISTINCT LEFT(municipio_ine, 2)) AS n FROM bronze_mitma_zone_mapping WHERE seccion_ine <> 'NA' GROUP BY 1)
        UNION ALL
        SELECT 'GAU', MAX(n)
        FROM (SELECT gau_mitma, COUNT(DISTINCT LEFT(municipio_ine, 2)) AS n FROM bronze_mitma_zone_mapping WHERE seccion_ine <> 'NA' GROUP BY 1)
    """)

zone_levelVARCHAR,max_provincesBIGINT
distritos,1
municipios,1
GAU,1


Each zone is in one province only, nice.

**The region (CCAA).** The holidays are per region, so we need to know the region of each province, and that is in `bronze_ine_provinces`. At first we wrote this list by hand, but then we found the INE page and the groups of provinces were the same, so now it comes from a source. First we had the region code and name inside the provinces table, but that repeats the name in each province. It also meant that the holidays couldn't have a real foreign key, because a code repeated in several provinces can't be a key. So regions get their own small table, `silver_dim_ccaa`.

In [49]:
con.sql("""
        CREATE TABLE silver_dim_ccaa AS
        SELECT DISTINCT CODAUTO AS ccaa_code, "Comunidad Autónoma" AS ccaa
        FROM bronze_ine_provinces
        ORDER BY 1
    """)

con.sql("""
        CREATE TABLE silver_dim_province AS
        SELECT CPRO AS province_code, Provincia AS province_name, CODAUTO AS ccaa_code
        FROM bronze_ine_provinces
        ORDER BY 1
    """)

con.sql("""
        SELECT (SELECT COUNT(*) FROM silver_dim_province) AS provinces, (SELECT COUNT(*) FROM silver_dim_ccaa) AS regions
    """)

provincesBIGINT,regionsBIGINT
52,19


Now we can build the zone dimension.

In [50]:
con.sql("""
        CREATE TABLE silver_dim_zone AS
        WITH provinces AS (
            SELECT DISTINCT distrito_mitma, municipio_mitma, gau_mitma, LEFT(municipio_ine, 2) AS province_code
            FROM bronze_mitma_zone_mapping WHERE seccion_ine <> 'NA'
        ),
        info AS (
            SELECT 'distritos' AS zone_level, distrito_mitma AS zone_id, ANY_VALUE(province_code) AS province_code FROM provinces GROUP BY 2
            UNION ALL SELECT 'municipios', municipio_mitma, ANY_VALUE(province_code) FROM provinces GROUP BY 2
            UNION ALL SELECT 'GAU', gau_mitma, ANY_VALUE(province_code) FROM provinces GROUP BY 2
        ),
        geo AS (
            SELECT zone_level, ID AS zone_id,
                   ANY_VALUE(geom) FILTER (geometry_type = 'polygon') AS geom,
                   ANY_VALUE(geom) FILTER (geometry_type = 'centroid') AS centroid
            FROM bronze_mitma_zone_geometries GROUP BY 1, 2
        ),
        all_zones AS (
            SELECT zone_level, zone_id FROM geo
            UNION SELECT zone_level, 'externo' FROM (VALUES ('distritos'), ('municipios'), ('GAU')) t(zone_level)
        )
        SELECT a.zone_level, a.zone_id,
               COALESCE(n.name, a.zone_id) AS zone_name,
               i.province_code,
               a.zone_id LIKE 'FR%' OR a.zone_id LIKE 'PT%' OR a.zone_id = 'externo' AS is_foreign,
               g.geom, g.centroid
        FROM all_zones a
        LEFT JOIN bronze_mitma_zone_names n ON a.zone_level = n.zone_level AND a.zone_id = n.ID
        LEFT JOIN info i ON a.zone_level = i.zone_level AND a.zone_id = i.zone_id
        LEFT JOIN geo g ON a.zone_level = g.zone_level AND a.zone_id = g.zone_id
    """)

con.sql("""
        SELECT zone_level, COUNT(*) AS zones, COUNT(*) FILTER (is_foreign) AS foreign_zones,
               COUNT(*) FILTER (province_code IS NULL AND NOT is_foreign) AS no_province
        FROM silver_dim_zone GROUP BY 1
    """)

zone_levelVARCHAR,zonesBIGINT,foreign_zonesBIGINT,no_provinceBIGINT
GAU,2204,118,49
distritos,3910,118,49
municipios,2736,118,49


## 3.2 Joining INE and MITMA by zone

This is the hard part. MITMA zones are not INE zones: INE has about 8,100 municipalities and MITMA 2,618 "municipalities" (the small ones are grouped). And INE gives us `Población` (`P_i`) and income, which must end up in MITMA zones. We always check against the total we found in 1.4: **47,972,964 people**.

**Attempt 1: join the codes directly.** Both sides have 5-digit municipality codes, so why not?

In [51]:
con.sql("""
        CREATE MACRO num(x) AS TRY_CAST(REPLACE(REPLACE(x, '.', ''), ',', '.') AS double)
    """)
con.sql("""
        CREATE TEMP TABLE ine_pop_municipality AS
            SELECT LEFT(Municipios, 5) AS municipality_ine, num(Total) AS population
            FROM bronze_ine_adrh WHERE indicador = 'Población' AND Periodo = '2023' AND Distritos IS NULL AND Secciones IS NULL
    """)

con.sql("""
        SELECT SUM(population) AS total, COUNT(*) AS municipalities,
               COUNT(z.zone_id) AS matched, SUM(population) FILTER (z.zone_id IS NOT NULL) AS population_matched,
               ROUND(100 - 100 * SUM(population) FILTER (z.zone_id IS NOT NULL) / SUM(population), 1) AS pct_lost
        FROM ine_pop_municipality i LEFT JOIN silver_dim_zone z ON z.zone_level = 'municipios' AND i.municipality_ine = z.zone_id
    """)

totalDOUBLE,municipalitiesBIGINT,matchedBIGINT,population_matchedDOUBLE,pct_lostDOUBLE
47972964.0,8139,1645,42691198.0,11.0


We lose 11% of the population and 80% of the municipalities: all the small towns inside `_AM` zones. It doesn't even raise an error, the data just disappears. Direct join discarded.

**Attempt 2: use the MITMA relation file at municipality level.** We take the distinct pairs (INE municipality, MITMA municipality) and we sum.

In [52]:
con.sql("""
        CREATE TEMP TABLE pairs_municipality AS
            SELECT DISTINCT municipio_ine, municipio_mitma FROM bronze_mitma_zone_mapping WHERE seccion_ine <> 'NA'
    """)

con.sql("""
        SELECT SUM(population) AS total, 47972964 - SUM(population) AS difference
        FROM ine_pop_municipality JOIN pairs_municipality ON municipality_ine = municipio_ine
    """)

totalDOUBLE,differenceDOUBLE
47974050.0,-1086.0


The total is close but it is **higher** than the real one, so something is counted twice. It must be Los Montesinos (03903) from section 1.3, which has two MITMA zones, so the join duplicates its people. But something could also be missing without us noticing, so we look for INE municipalities that aren't in the file:

In [53]:
con.sql("""
        SELECT i.municipality_ine, i.population, (SELECT LIST(DISTINCT Municipios) FROM bronze_ine_adrh b WHERE LEFT(b.Municipios, 5) = i.municipality_ine) AS name
        FROM ine_pop_municipality i WHERE municipality_ine NOT IN (SELECT municipio_ine FROM pairs_municipality) ORDER BY population desc nulls last
    """)

municipality_ineVARCHAR,populationDOUBLE,nameVARCHAR[]
48916,4556.0,['48916 Usansolo']
15063,NULL,['15063 Oza dos Ríos']
15026,NULL,['15026 Cesuras']
04039,NULL,['04039 Darrical']
12066,NULL,['12066 Gatova']
17122,NULL,['17122 Palmerola']
36011,NULL,['36011 Cerdedo']
36012,NULL,['36012 Cotobade']


In [54]:
con.sql("""
        SELECT population, (SELECT COUNT(*) FROM pairs_municipality WHERE municipio_ine = '03903') AS zones_of_montesinos
        FROM ine_pop_municipality WHERE municipality_ine = '03903'
    """)

populationDOUBLE,zones_of_montesinosBIGINT
5642.0,2


Seven of these have no population in 2023, so these codes don't exist any more. The one that matters is Usansolo (48916), with 4,556 people. We looked it up: it split from Galdakao (48036) in 2022, so the relation file, which is older, doesn't have it.

And now the numbers fit: +5,642 from Los Montesinos counted twice, -4,556 from Usansolo = +1,086, which is exactly the difference we got. Two errors were hiding each other, and the total alone looked almost right.

But even if we fixed those two cases, there is a bigger problem with this attempt: **it only works for municipalities**. A big city like Madrid is one INE municipality and many MITMA districts, so municipality rows can't give us the population of a district.

**Attempt 3: go down to census sections.** The section is the smallest unit, and each one goes to only one MITMA district (checked in 1.3). From there we can sum to any of the three levels with the same query. That's what the relation file is for.

In [55]:
con.sql("""
        CREATE TEMP TABLE ine_pop_section AS
            SELECT LEFT(Secciones, 10) AS section_id, LEFT(Secciones, 7) AS district_ine, LEFT(Secciones, 5) AS municipality_ine, num(Total) AS population
            FROM bronze_ine_adrh WHERE indicador = 'Población' AND Periodo = '2023' AND Secciones IS NOT NULL
    """)

con.sql("""
        SELECT SUM(population) AS total, SUM(population) FILTER (m.seccion_ine IS NOT NULL) AS population_matched,
               SUM(population) FILTER (m.seccion_ine IS NULL) AS population_lost,
               ROUND(100 * SUM(population) FILTER (m.seccion_ine IS NULL) / SUM(population), 2) AS pct_lost,
               COUNT(*) FILTER (m.seccion_ine IS NULL) AS sections_not_matched,
               COUNT(*) FILTER (m.seccion_ine IS NULL AND population IS NOT NULL) AS of_which_with_people
        FROM ine_pop_section s LEFT JOIN bronze_mitma_zone_mapping m ON section_id = seccion_ine
    """)

totalDOUBLE,population_matchedDOUBLE,population_lostDOUBLE,pct_lostDOUBLE,sections_not_matchedBIGINT,of_which_with_peopleBIGINT
47972964.0,47313730.0,659234.0,1.37,896,402


No duplicates now, but we lose 659,234 people (1.4%). It doesn't sound like much, but it isn't spread evenly:

In [56]:
con.sql("""
        SELECT municipality_ine, ANY_VALUE(z.zone_name) AS name, COUNT(*) AS sections_lost, SUM(population) AS people_lost
        FROM ine_pop_section s LEFT JOIN silver_dim_zone z ON z.zone_level = 'municipios' AND z.zone_id = municipality_ine
        WHERE section_id NOT IN (SELECT seccion_ine FROM bronze_mitma_zone_mapping) AND population IS NOT NULL
        GROUP BY 1 ORDER BY people_lost desc LIMIT 6
    """)

municipality_ineVARCHAR,nameVARCHAR,sections_lostBIGINT,people_lostDOUBLE
28079,Madrid,34,69498.0
46070,Bétera,12,19327.0
50297,Zaragoza,13,19305.0
28123,Rivas-Vaciamadrid,10,18319.0
28022,Boadilla del Monte,5,10902.0
28005,Alcalá de Henares,5,10767.0


Madrid, Zaragoza and growing towns like Bétera or Rivas, so some zones would have much less population than they really have. Why are these sections not in the file? Maybe they are new ones. Let's see when each lost section appears in the INE data.

In [57]:
con.sql("""
        WITH lost AS (SELECT section_id FROM ine_pop_section WHERE section_id NOT IN (SELECT seccion_ine FROM bronze_mitma_zone_mapping) AND population IS NOT NULL)
        SELECT first_year, COUNT(*) AS sections FROM (
            SELECT LEFT(Secciones, 10) AS section_id, MIN(Periodo) FILTER (num(Total) IS NOT NULL) AS first_year
            FROM bronze_ine_adrh WHERE indicador = 'Población' AND Secciones IS NOT NULL AND LEFT(Secciones, 10) IN (SELECT section_id FROM lost) GROUP BY 1
        ) GROUP BY 1 ORDER BY 1
    """)

first_yearVARCHAR,sectionsBIGINT
2015,1
2021,102
2022,200
2023,99


Almost all of them first appear in 2021, 2022 or 2023 (only one is older). So they are new sections, which makes sense because INE redraws sections when they grow too much, and the relation file doesn't have them because it was made with an older list. The rest of the unmatched sections (the ones with no people in 2023) are probably the old ones they replaced, like the one we saw in 1.4.

**Attempt 4: sections with a fallback.** A new section has a new code, but the code includes its INE district and municipality (`MMMMM DD SSS`), and those usually do exist in the file. So the idea is:

1. If the section is in the file, use it (`exact`).
2. If not, look for its INE district and use the MITMA district where most of the sections of that INE district go (`by_district`).
3. If the INE district is new too, do the same with the municipality (`by_municipality`).
4. Usansolo, by hand, to Galdakao (`manual`).

The "most sections" rule in steps 2 and 3 is not perfect: a new section could be in another MITMA district of the same municipality. But the alternative was losing those people. We keep how it was matched in a column, so anyone can see or filter it. Each MITMA district goes to only one municipality and GAU (checked in 1.3), so the district is enough to know the other two levels.

We keep one row per section and level, so we can sum to the three levels with the same query.

In [58]:
con.sql("""
        CREATE TEMP TABLE section_zone AS
        WITH mapping AS (SELECT * FROM bronze_mitma_zone_mapping WHERE seccion_ine <> 'NA'),
        by_district AS (SELECT distrito_ine, MODE(distrito_mitma) AS district_id FROM mapping GROUP BY 1),
        by_municipality AS (SELECT municipio_ine, MODE(distrito_mitma) AS district_id FROM mapping GROUP BY 1),
        matched AS (
            SELECT s.section_id, s.municipality_ine,
                   COALESCE(m.distrito_mitma, d.district_id, b.district_id, CASE WHEN s.municipality_ine = '48916' THEN '48036' END) AS district_id,
                   CASE WHEN m.distrito_mitma IS NOT NULL THEN 'exact'
                        WHEN d.district_id IS NOT NULL THEN 'by_district'
                        WHEN b.district_id IS NOT NULL THEN 'by_municipality'
                        WHEN s.municipality_ine = '48916' THEN 'manual' END AS match_type
            FROM ine_pop_section s
            LEFT JOIN mapping m ON s.section_id = m.seccion_ine
            LEFT JOIN by_district d ON s.district_ine = d.distrito_ine
            LEFT JOIN by_municipality b ON s.municipality_ine = b.municipio_ine
            WHERE s.population IS NOT NULL
        )
        SELECT m.section_id, 'distritos' AS zone_level, m.district_id AS zone_id, m.match_type FROM matched m
        UNION ALL
        SELECT m.section_id, 'municipios', z.municipio_mitma, m.match_type FROM matched m JOIN (SELECT DISTINCT distrito_mitma, municipio_mitma, gau_mitma FROM mapping) z ON m.district_id = z.distrito_mitma
        UNION ALL
        SELECT m.section_id, 'GAU', z.gau_mitma, m.match_type FROM matched m JOIN (SELECT DISTINCT distrito_mitma, municipio_mitma, gau_mitma FROM mapping) z ON m.district_id = z.distrito_mitma
    """)

con.sql("""
        SELECT match_type, COUNT(*) AS sections, SUM(population) AS population, ROUND(100 * SUM(population) / 47972964, 3) AS pct
        FROM section_zone JOIN ine_pop_section USING (section_id) WHERE zone_level = 'distritos' GROUP BY 1 ORDER BY 2 DESC
    """)

match_typeVARCHAR,sectionsBIGINT,populationDOUBLE,pctDOUBLE
exact,36075,47313730.0,98.626
by_district,398,653273.0,1.362
manual,3,4556.0,0.009
by_municipality,1,1405.0,0.003


In [59]:
con.sql("""
        SELECT zone_level, SUM(population) AS total, 47972964 - SUM(population) AS difference, COUNT(*) - COUNT(DISTINCT section_id) AS repeated_sections
        FROM section_zone JOIN ine_pop_section USING (section_id)
        GROUP BY 1
    """)

zone_levelVARCHAR,totalDOUBLE,differenceDOUBLE,repeated_sectionsBIGINT
GAU,47972964.0,0.0,0
municipios,47972964.0,0.0,0
distritos,47972964.0,0.0,0


Now the total is exactly right, with no section repeated, and 98.6% of the people are matched `exact`. This is the mapping we use. In the end we don't keep it as a Silver table: we only need it to build `silver_zone_metrics` and no query looks at it afterwards, so it is a temporary table here. The numbers above stay as the record of how it was matched. This INE population is `P_i`, as the project asks.

## 3.3 Income: how do you add it up?

Population can be summed, but average income can't. For a zone with several sections, the first thing we thought of was a simple average of the sections. But a section with 500 people would count the same as one with 3,000. So we compare it to a mean weighted by population. And we can test both, because INE publishes the official income per municipality.

In [60]:
con.sql("""
        CREATE TEMP TABLE section_income AS
            SELECT i.section_id, i.municipality_ine, i.population, num(Total) AS income
            FROM bronze_ine_adrh b JOIN ine_pop_section i ON LEFT(b.Secciones, 10) = i.section_id
            WHERE b.indicador = 'Renta neta media por persona' AND b.Periodo = '2023'
    """)

con.sql("""
        WITH calc AS (
            SELECT municipality_ine, COUNT(*) AS n_sections, AVG(income) AS simple_avg,
                   SUM(income * population) FILTER (income IS NOT NULL) / SUM(population) FILTER (income IS NOT NULL) AS weighted_avg
            FROM section_income GROUP BY 1
        ),
        official AS (
            SELECT LEFT(Municipios, 5) AS municipality_ine, num(Total) AS official FROM bronze_ine_adrh
            WHERE indicador = 'Renta neta media por persona' AND Periodo = '2023' AND Distritos IS NULL AND Secciones IS NULL
        )
        SELECT COUNT(*) AS municipalities_with_several_sections,
               ROUND(AVG(ABS(simple_avg - official) / official) * 100, 2) AS simple_avg_error_pct,
               ROUND(AVG(ABS(weighted_avg - official) / official) * 100, 2) AS weighted_avg_error_pct,
               ROUND(MAX(ABS(weighted_avg - official) / official) * 100, 1) AS weighted_worst_error_pct
        FROM calc JOIN official USING (municipality_ine) WHERE n_sections > 1 AND official IS NOT NULL AND weighted_avg IS NOT NULL
    """)

municipalities_with_several_sectionsBIGINT,simple_avg_error_pctDOUBLE,weighted_avg_error_pctDOUBLE,weighted_worst_error_pctDOUBLE
2321,0.85,0.08,14.9


The weighted mean is clearly closer, so we use that (and it is what income per person should mathematically be). Nulls have to be handled too: if a section has no income, its people shouldn't be in the denominator, otherwise the zone looks poorer. So we use a small macro for this.

The worst case is high, so we have a look at it before moving on:

In [61]:
con.sql("""
        WITH calc AS (
            SELECT municipality_ine, COUNT(*) AS n_sections, COUNT(income) AS n_with_income, SUM(population) AS population,
                   SUM(income * population) FILTER (income IS NOT NULL) / SUM(population) FILTER (income IS NOT NULL) AS weighted_avg
            FROM section_income GROUP BY 1
        ),
        official AS (
            SELECT LEFT(Municipios, 5) AS municipality_ine, Municipios AS name, num(Total) AS official FROM bronze_ine_adrh
            WHERE indicador = 'Renta neta media por persona' AND Periodo = '2023' AND Distritos IS NULL AND Secciones IS NULL
        )
        SELECT name, n_sections, n_with_income, population, official, ROUND(weighted_avg) AS weighted_avg,
               ROUND(100 * ABS(weighted_avg - official) / official, 1) AS pct_error
        FROM calc JOIN official USING (municipality_ine) WHERE n_sections > 1 ORDER BY pct_error desc nulls last LIMIT 5
    """)

nameVARCHAR,n_sectionsBIGINT,n_with_incomeBIGINT,populationDOUBLE,officialDOUBLE,weighted_avgDOUBLE,pct_errorDOUBLE
08108 Lliçà de Vall,4,3,6803.0,15754.0,18104.0,14.9
31006 Ablitas,2,2,2659.0,13588.0,14634.0,7.7
28115 Pozuelo de Alarcón,57,56,87407.0,30524.0,28299.0,7.3
35021 Santa Brígida,12,11,18209.0,17397.0,18579.0,6.8
28006 Alcobendas,67,66,119536.0,21644.0,20297.0,6.2


The worst one is Lliçà de Vall, where one of the 4 sections has no income, so it makes sense that we can't reach the official figure. But Ablitas has all its sections and is still 7.7% off, so the official figure isn't exactly a population-weighted mean. We don't know why (maybe the income tables use a slightly different population). As the average error is 0.08%, we accept it, but it goes on our list of things to check.

Not all 20 indicators are needed. We keep the ones that help describe a zone for BQ2 and BQ3:

| Keep | Why |
|---|---|
| `Población` | It's `P_i` |
| `Renta neta media por persona` | The income measure, and the project asks to include income |
| `Fuente de ingreso: salario, pensiones, prestaciones por desempleo` | Tell us if a zone is working, retired, or unemployed |
| `Índice de Gini` | To relate inequality and lack of transport. A weighted mean of Gini isn't exact, so it is an approximation |
| `Edad media`, `% 65 y más` | Who lives in the zone |

We don't keep the medians: you can't get a median of a zone from the medians of its sections.

## 3.4 Businesses (`E_j`)

DIRCE only exists per municipality. So here we have to use the municipality pairs from attempt 2, removing the wrong Los Montesinos row by hand. The municipality has only one code, and we can't divide its businesses, so we keep it in its own zone `03903`.

**Districts**: at first we thought of dividing the businesses of Madrid between its districts according to population. But that would be inventing data: offices aren't where people live, and that is exactly what `E_j` should measure. So businesses are `NULL` at district level, and BQ2 is done with municipalities.

**Which groups**: we saw in 1.5 that the detailed groups are empty in many towns. Let's see what happens when we sum them to MITMA zones.

In [62]:
con.sql("""
        CREATE TEMP TABLE dirce_zone AS
            SELECT p.municipio_mitma AS zone_id, "Grupos CNAE" AS grp, SUM(num(Total)) AS businesses
            FROM bronze_ine_dirce d JOIN pairs_municipality p ON LEFT(Municipios, 5) = municipio_ine
            WHERE Periodo = '2023' AND NOT (municipio_ine = '03903' AND municipio_mitma = '03015')
            GROUP BY 1, 2
    """)
con.sql("""
        SELECT grp, COUNT(*) AS zones, COUNT(businesses) AS zones_with_value, SUM(businesses) AS total
        FROM dirce_zone GROUP BY 1 ORDER BY zones_with_value desc
    """)

grpVARCHAR,zonesBIGINT,zones_with_valueBIGINT,totalDOUBLE
Total,2569,2569,3205679.0
"Comercio, transporte y hostelería",2569,2504,1071915.0
Total servicios,2569,2501,1522044.0
F Construcción,2569,2479,360796.0
Industria,2569,2474,168331.0
MN Actividades profesionales y técnicas,2569,1313,559767.0
Otros servicios personales,2569,1313,296293.0
"Educación, sanidad y servicios sociales",2569,1313,258863.0
L Actividades inmobiliarias,2569,1236,182669.0
K Actividades financieras y de seguros,2569,1136,68386.0


All 2,569 inhabited zones have `Total` and its sum (3,205,679) is 99.9% of the national total (3,207,580). The four big groups (`Comercio...`, `Total servicios`, `F Construcción`, `Industria`) exist in more than 96% of the zones. The others only in about half. So we keep `Total` as `E_j`, plus those four, and not the rest. Empty is `NULL`, not zero, because we can't say there are no businesses, only that INE doesn't give the number.

Also, we looked at the DIRCE municipalities that aren't in the relation file (old merged municipalities and Usansolo). All of them are empty in 2023, so nothing is lost:

In [63]:
con.sql("""
        SELECT Municipios, Total FROM bronze_ine_dirce
        WHERE Periodo = '2023' AND "Grupos CNAE" = 'Total' AND Municipios IS NOT NULL AND LEFT(Municipios, 5) NOT IN (SELECT municipio_ine FROM pairs_municipality)
    """)

MunicipiosVARCHAR,TotalVARCHAR
17122 Palmerola,NULL
15026 Cesuras,NULL
15063 Oza dos Ríos,NULL
36011 Cerdedo,NULL
36012 Cotobade,NULL
48916 Usansolo,NULL


Now we can build the metrics table, using the same query for the three levels.

In [64]:
con.sql("""
        CREATE MACRO wavg(x, p) AS SUM(x * p) FILTER (x IS NOT NULL) / SUM(p) FILTER (x IS NOT NULL)
    """)

con.sql("""
        CREATE TABLE silver_zone_metrics AS
        WITH section_values AS (
            SELECT LEFT(Secciones, 10) AS section_id,
                   MAX(num(Total)) FILTER (indicador = 'Población') AS population,
                   MAX(num(Total)) FILTER (indicador = 'Renta neta media por persona') AS income,
                   MAX(num(Total)) FILTER (indicador = 'Fuente de ingreso: salario') AS wages,
                   MAX(num(Total)) FILTER (indicador = 'Fuente de ingreso: pensiones') AS pensions,
                   MAX(num(Total)) FILTER (indicador = 'Fuente de ingreso: prestaciones por desempleo') AS unemployment,
                   MAX(num(Total)) FILTER (indicador = 'Índice de Gini') AS gini,
                   MAX(num(Total)) FILTER (indicador = 'Edad media de la población') AS age,
                   MAX(num(Total)) FILTER (indicador = 'Porcentaje de población de 65 y más años') AS over_65
            FROM bronze_ine_adrh WHERE Periodo = '2023' AND Secciones IS NOT NULL GROUP BY 1
        ),
        ine AS (
            SELECT zone_level, zone_id, SUM(population) AS population, wavg(income, population) AS mean_income_per_person,
                   wavg(wages, population) AS pct_income_wages, wavg(pensions, population) AS pct_income_pensions, wavg(unemployment, population) AS pct_income_unemployment,
                   wavg(gini, population) AS gini, wavg(age, population) AS mean_age, wavg(over_65, population) AS pct_over_65
            FROM section_zone JOIN section_values USING (section_id) GROUP BY 1, 2
        ),
        business AS (
            SELECT zone_id,
                   MAX(businesses) FILTER (grp = 'Total')::INT AS businesses_total,
                   MAX(businesses) FILTER (grp = 'Industria')::INT AS businesses_industry,
                   MAX(businesses) FILTER (grp = 'F Construcción')::INT AS businesses_construction,
                   MAX(businesses) FILTER (grp = 'Comercio, transporte y hostelería')::INT AS businesses_trade_transport_hospitality,
                   MAX(businesses) FILTER (grp = 'Total servicios')::INT AS businesses_services
            FROM dirce_zone GROUP BY 1
        ),
        business_levels AS (
            SELECT 'municipios' AS zone_level, * FROM business
            UNION ALL
            SELECT 'GAU', gau_id, SUM(businesses_total)::INT, SUM(businesses_industry)::INT, SUM(businesses_construction)::INT, SUM(businesses_trade_transport_hospitality)::INT, SUM(businesses_services)::INT
            FROM business JOIN (SELECT DISTINCT municipio_mitma, gau_mitma AS gau_id FROM bronze_mitma_zone_mapping) ON zone_id = municipio_mitma GROUP BY 2
        )
        SELECT * FROM ine LEFT JOIN business_levels USING (zone_level, zone_id)
    """)

con.sql("""
        SELECT zone_level, COUNT(*) AS zones, SUM(population) AS population, COUNT(mean_income_per_person) AS with_income, COUNT(gini) AS with_gini,
               COUNT(businesses_total) AS with_businesses, SUM(businesses_total) AS businesses
        FROM silver_zone_metrics GROUP BY 1
    """)

zone_levelVARCHAR,zonesBIGINT,populationDOUBLE,with_incomeBIGINT,with_giniBIGINT,with_businessesBIGINT,businessesHUGEINT
municipios,2569,47972964.0,2568,2566,2569,3205679
GAU,2037,47972964.0,2037,2037,2037,3205679
distritos,3742,47972964.0,3741,3739,0,NULL


The population is 47,972,964 in the three levels. The 49 unpopulated lands aren't here, as they have no INE data.

## 3.5 Dates and holidays

**Attempt 1**: `silver_dim_date` with a column `is_holiday`. But a day like 2 May is a holiday only in Madrid, so one column per day isn't enough.

**Attempt 2**: a table `silver_holidays` per day and region, which joins with zones through `silver_dim_province`, plus `silver_dim_date` with `year`, `month`, `day`, `day_name`, `is_weekend` and `is_national_holiday`. Then we looked at those columns and all of them can be calculated: the first ones from the date with SQL functions, and `is_national_holiday` from `silver_holidays`.

**Final**: only `silver_holidays`, without a date dimension. It would not add any data.

The BOE has the region name but no code, and it doesn't write the names like INE (`Com. Madrid` and `Madrid, Comunidad de`). First we translated the 8 different names so they would match, but joining by a text name felt fragile. So we add the INE code (`CODAUTO`) to each holiday with a small list of the 19 BOE names, and we join by code.

How do we define national? We could use the `*` mark, but looking at the table, Epifanía has `**` (it could be replaced but nobody did) and it's a holiday all over Spain. So national means **it's a holiday in all 19 regions**.

In [65]:
con.sql("""
        CREATE TABLE silver_holidays AS
            SELECT MAKE_DATE(year::INT, month(strptime(m, '%B')), regexp_extract(fiesta, '^[0-9]+')::INT) AS date,
                   ccaa_code,
                   TRIM(regexp_replace(fiesta, '^[0-9]+', '')) AS holiday_name
            FROM bronze_boe_holidays b JOIN (VALUES
                ('ENERO','January'),('FEBRERO','February'),('MARZO','March'),('ABRIL','April'),('MAYO','May'),('JUNIO','June'),
                ('JULIO','July'),('AGOSTO','August'),('SEPTIEMBRE','September'),('OCTUBRE','October'),('NOVIEMBRE','November'),('DICIEMBRE','December')) t(mes, m) USING (mes)
            LEFT JOIN (VALUES
                ('Andalucía', '01'), ('Aragón', '02'), ('Asturias', '03'), ('Illes Balears', '04'),
                ('Canarias', '05'), ('Cantabria', '06'), ('Castilla y León', '07'), ('Castilla-La Mancha', '08'),
                ('Cataluña', '09'), ('Comunitat Valenciana', '10'), ('Extremadura', '11'), ('Galicia', '12'),
                ('Com. Madrid', '13'), ('Región Murcia', '14'), ('C. Foral Navarra', '15'), ('País Vasco', '16'),
                ('La Rioja', '17'), ('Ciudad de Ceuta', '18'), ('Ciudad de Melilla', '19')
            ) r(boe_name, ccaa_code) ON TRIM(regexp_replace(b.ccaa, '\\([0-9]\\)', '')) = boe_name
            WHERE marca <> ''
    """)

con.sql("""
        SELECT date, dayname(date) AS day_name FROM silver_holidays GROUP BY 1 HAVING COUNT(DISTINCT ccaa_code) = 19 ORDER BY 1
    """)

dateDATE,day_nameVARCHAR
2023-01-06,Friday
2023-04-07,Friday
2023-05-01,Monday
2023-08-15,Tuesday
2023-10-12,Thursday
2023-11-01,Wednesday
2023-12-06,Wednesday
2023-12-08,Friday
2023-12-25,Monday


In [66]:
con.sql("""
        SELECT (SELECT COUNT(*) FROM silver_holidays WHERE ccaa_code IS NULL OR ccaa_code NOT IN (SELECT ccaa_code FROM silver_dim_ccaa)) AS holidays_without_region_code,
               (SELECT COUNT(*) FROM silver_holidays WHERE date BETWEEN '2023-03-06' AND '2023-03-12') AS holidays_in_our_sample_week
    """)

holidays_without_region_codeBIGINT,holidays_in_our_sample_weekBIGINT
0,0


All the holidays got their region code. Our sample week has no holidays, so BQ1 can't be tested with holidays until we have more data.

## 3.6 Fact tables

**Trips.** This table changed a lot from our first version, which had almost every column of the file:

- `fecha` and `periodo` become one column, `trip_datetime`, instead of a date and an hour.
- We removed `distancia`. The professor told us we don't need it in Silver, because we can estimate the distance from the origin and destination zones (from the centroids of the zones).
- We also removed `estudio_*`, `viajes_km` and the traveller profile (`residencia`, `renta`, `edad`, `sexo`), because none of our questions uses them. They all stay in Bronze if we ever need them.
- We keep the activities, because they let us tell apart for example home to work trips.
- We don't remove foreign trips or trips inside a zone, because they are real. The analysis can exclude them when needed (like in the gravity model).

Without those columns the same combination appears in several rows, so we have to sum `viajes`. We check below that no trips are lost.


**Overnight stays.** First version: `residence_zone_id` and `overnight_zone_id`, both with the level of the file. Section 1.2 showed it's wrong, so now it's `residence_district_id` (always a district) and `overnight_zone_id` (level of the file). The foreign key of the residence is always to (`distritos`, id).

In [67]:
con.sql("""
        CREATE TABLE silver_fact_trips AS
            SELECT strptime(fecha || periodo, '%Y%m%d%H') AS trip_datetime, zone_level,
                   origen AS origin_zone_id, destino AS destination_zone_id,
                   actividad_origen AS origin_activity, actividad_destino AS destination_activity,
                   SUM(viajes::DOUBLE) AS trips
            FROM bronze_mitma_od_daily
            GROUP BY ALL
    """)

con.sql("""
        CREATE TABLE silver_fact_overnight AS
            SELECT strptime(fecha, '%Y%m%d')::DATE AS date, zone_level, zona_residencia AS residence_district_id,
                   zona_pernoctacion AS overnight_zone_id, personas::DOUBLE AS people
            FROM bronze_mitma_overnight_daily
    """)

con.sql("""
        SELECT * FROM silver_fact_trips LIMIT 3
    """)

trip_datetimeTIMESTAMP,zone_levelVARCHAR,origin_zone_idVARCHAR,destination_zone_idVARCHAR,origin_activityVARCHAR,destination_activityVARCHAR,tripsDOUBLE
2023-03-06 19:00:00,distritos,0821704,08221,frecuente,casa,96.421
2023-03-06 15:00:00,distritos,0801901,08221,frecuente,casa,6.053
2023-03-06 12:00:00,distritos,0801903,08221,frecuente,casa,4.454


In [68]:
con.sql("""
        SELECT (SELECT COUNT(*) FROM bronze_mitma_od_daily) AS rows_bronze, COUNT(*) AS rows_silver,
               ROUND(SUM(trips) - (SELECT SUM(viajes::DOUBLE) FROM bronze_mitma_od_daily), 3) AS trips_difference
        FROM silver_fact_trips
    """)

rows_bronzeBIGINT,rows_silverBIGINT,trips_differenceDOUBLE
18267798,8901953,-0.0


**Distances.** `d_ij` isn't given by MITMA, so we compute it between centroids. Our first idea was `st_distance` in metres because the coordinates are projected. But the Canary Islands are far from UTM zone 30, so we were afraid of the error. MITMA's dictionary says its distance bands are great-circle distance ("ortodrómica"), so we compare both with some known cities.

In [69]:
con.sql("""
        WITH c AS (
            SELECT zone_id, zone_name, centroid, st_transform(centroid, 'EPSG:4326', true) AS lonlat
            FROM silver_dim_zone WHERE zone_level = 'municipios' AND zone_id IN ('28079', '08019', '41091', '35016', '38038')
        )
        SELECT a.zone_name AS origin, b.zone_name AS destination, ROUND(st_distance(a.centroid, b.centroid) / 1000) AS km_planar,
               ROUND(st_distance_sphere(a.lonlat, b.lonlat) / 1000) AS km_sphere
        FROM c a JOIN c b ON a.zone_id < b.zone_id ORDER BY km_sphere
    """)

originVARCHAR,destinationVARCHAR,km_planarDOUBLE,km_sphereDOUBLE
"Palmas de Gran Canaria, Las",Santa Cruz de Tenerife,91.0,89.0
Madrid,Sevilla,393.0,394.0
Barcelona,Madrid,503.0,502.0
Barcelona,Sevilla,828.0,826.0
"Palmas de Gran Canaria, Las",Sevilla,1371.0,1362.0
Santa Cruz de Tenerife,Sevilla,1384.0,1374.0
Madrid,"Palmas de Gran Canaria, Las",1756.0,1746.0
Madrid,Santa Cruz de Tenerife,1764.0,1752.0
Barcelona,"Palmas de Gran Canaria, Las",2188.0,2178.0
Barcelona,Santa Cruz de Tenerife,2207.0,2195.0


The difference is small, at most around 2% in the Canaries and less on the mainland, so it isn't very important for us. Even so, we use the sphere: it is the same definition MITMA uses, and it's one more function call.

Then, where do we put it? Our first design had a table `silver_zone_distances` with all the pairs of zones, about 14 million rows at district level. But the distance never changes and it comes from the centroids, which are already in `silver_dim_zone`, so let's see how long it takes to calculate all the pairs at municipality level, the level of BQ2.

In [70]:
%%time
con.sql("""
        WITH c AS (
            SELECT zone_id, st_transform(centroid, 'EPSG:4326', true) AS lonlat
            FROM silver_dim_zone WHERE zone_level = 'municipios' AND NOT is_foreign
        )
        SELECT COUNT(*) AS pairs, ROUND(MIN(distance_km), 2) AS min_km, ROUND(MAX(distance_km)) AS max_km
        FROM (SELECT st_distance_sphere(a.lonlat, b.lonlat) / 1000 AS distance_km FROM c a JOIN c b ON a.zone_id <> b.zone_id)
    """).df()

CPU times: user 444 ms, sys: 66 μs, total: 444 ms
Wall time: 350 ms


,pairs,min_km,max_km
0,6851306,0.22,2516.0


It is very fast, so we don't need to store it. Only BQ2 uses it, so we calculate it when we need it, and Silver only keeps the centroids. The project gives distances per zone pair as an example of a Silver table, but it's only an example. We leave out the pairs of a zone with itself, because the distance would be 0 and the gravity model would divide by zero, and we include the pairs with no trips, because for BQ2 they are an interesting result.

## 3.7 Final design and checks

| Table | Grain | Primary key | Foreign keys |
|---|---|---|---|
| `silver_dim_ccaa` | region | `ccaa_code` | |
| `silver_dim_province` | province | `province_code` | `ccaa_code` -> `silver_dim_ccaa` |
| `silver_dim_zone` | zone and level | `zone_level`, `zone_id` | `province_code` -> `silver_dim_province` |
| `silver_holidays` | day and region | `date`, `ccaa_code` | `ccaa_code` -> `silver_dim_ccaa` |
| `silver_zone_metrics` | zone and level (2023) | `zone_level`, `zone_id` | same columns -> `silver_dim_zone` |
| `silver_fact_trips` | hour, origin, destination and activities | all columns except `trips` | `zone_level` + origin and destination -> `silver_dim_zone` |
| `silver_fact_overnight` | day, residence, overnight zone | `date`, `zone_level`, `residence_district_id`, `overnight_zone_id` | (`distritos`, `residence_district_id`) and (`zone_level`, `overnight_zone_id`) -> `silver_dim_zone` |

DuckLake (Sprint 2) doesn't enforce primary or foreign keys, so instead of constraints we write queries that look for duplicates and orphans, which we can run after every load. They should all be 0, and they are (for this one day of data).

In [71]:
con.sql("""
        SELECT 'duplicates silver_dim_province' AS test, COUNT(*) - COUNT(DISTINCT province_code) AS problems FROM silver_dim_province
        UNION ALL SELECT 'duplicates silver_dim_zone', COUNT(*) - COUNT(DISTINCT (zone_level, zone_id)) FROM silver_dim_zone
        UNION ALL SELECT 'duplicates silver_holidays', COUNT(*) - COUNT(DISTINCT (date, ccaa_code)) FROM silver_holidays
        UNION ALL SELECT 'duplicates silver_dim_ccaa', COUNT(*) - COUNT(DISTINCT ccaa_code) FROM silver_dim_ccaa
        UNION ALL SELECT 'duplicates silver_zone_metrics', COUNT(*) - COUNT(DISTINCT (zone_level, zone_id)) FROM silver_zone_metrics
        UNION ALL SELECT 'duplicates silver_fact_overnight', COUNT(*) - COUNT(DISTINCT (date, zone_level, residence_district_id, overnight_zone_id)) FROM silver_fact_overnight
        UNION ALL SELECT 'orphans province region', COUNT(*) FROM silver_dim_province WHERE ccaa_code NOT IN (SELECT ccaa_code FROM silver_dim_ccaa)
        UNION ALL SELECT 'orphans holidays region', COUNT(*) FROM silver_holidays WHERE ccaa_code IS NULL OR ccaa_code NOT IN (SELECT ccaa_code FROM silver_dim_ccaa)
        UNION ALL SELECT 'orphans metrics', COUNT(*) FROM silver_zone_metrics m LEFT JOIN silver_dim_zone z USING (zone_level, zone_id) WHERE z.zone_id IS NULL
        UNION ALL SELECT 'orphans trips origin', COUNT(*) FROM silver_fact_trips t LEFT JOIN silver_dim_zone z ON z.zone_level = t.zone_level AND z.zone_id = t.origin_zone_id WHERE z.zone_id IS NULL
        UNION ALL SELECT 'orphans trips destination', COUNT(*) FROM silver_fact_trips t LEFT JOIN silver_dim_zone z ON z.zone_level = t.zone_level AND z.zone_id = t.destination_zone_id WHERE z.zone_id IS NULL
        UNION ALL SELECT 'orphans overnight residence', COUNT(*) FROM silver_fact_overnight o LEFT JOIN silver_dim_zone z ON z.zone_level = 'distritos' AND z.zone_id = o.residence_district_id WHERE z.zone_id IS NULL
        UNION ALL SELECT 'orphans overnight zone', COUNT(*) FROM silver_fact_overnight o LEFT JOIN silver_dim_zone z ON z.zone_level = o.zone_level AND z.zone_id = o.overnight_zone_id WHERE z.zone_id IS NULL
        UNION ALL SELECT 'zones with unknown province', COUNT(*) FROM silver_dim_zone WHERE province_code NOT IN (SELECT province_code FROM silver_dim_province)
        UNION ALL SELECT 'levels where population is not 47972964', COUNT(*) FROM (SELECT zone_level FROM silver_zone_metrics GROUP BY 1 HAVING SUM(population) <> 47972964)
        UNION ALL SELECT 'trips lost from Bronze to Silver', ABS(ROUND((SELECT SUM(trips) FROM silver_fact_trips) - (SELECT SUM(viajes::DOUBLE) FROM bronze_mitma_od_daily)))
    """).df()

,test,problems
0,duplicates silver_dim_province,0.0
1,duplicates silver_dim_zone,0.0
2,duplicates silver_holidays,0.0
3,duplicates silver_dim_ccaa,0.0
4,duplicates silver_zone_metrics,0.0
5,duplicates silver_fact_overnight,0.0
6,orphans province region,0.0
7,orphans holidays region,0.0
8,orphans metrics,0.0
9,orphans trips origin,0.0


### What changed from our first idea to the final design

| First idea | Final | What made us change |
|---|---|---|
| One big integrated table | Star schema | Copying zone data 18 million times a day |
| `zone_id` as key | `zone_level` + `zone_id` | Same IDs in several levels |
| Province = first 2 characters of the zone | From the INE municipality in the relation file | 49 zones ended up in Palencia, and GAU IDs are names |
| Direct join of municipality codes | Sections with a fallback | 11% of people lost, then 1.4%, now 0 |
| Simple average of income | Weighted, ignoring nulls | Closer to the official figure |
| Divide businesses between districts | `NULL` at district level | It would be invented data |
| Residence at the level of the file | Always a district | Checked against the zones and the dictionary |
| Date dimension with `is_holiday` | Only `silver_holidays` per region | Some holidays are only in one region, and the rest can be calculated |
| Table with the distance of all zone pairs | Calculated when needed from the centroids | It takes less than a second and only BQ2 uses it |
| Silver table with the section of each zone | A step when building `silver_zone_metrics` | Once the metrics exist, no query needs it |
| Every column of the trips file | Datetime, zones, activities and trips | Our questions don't use the rest, and it reduces the rows |

### Things we still have to do or check

- Read the INE methodology to confirm why income and DIRCE values are missing. So far we only know that they tend to be small places.
- Everything here is one day of trips. We have to check that the same holds for the other days.
- The official municipal income isn't exactly the weighted mean of its sections (Ablitas is 7.7% off even with all its sections).
- `by_district` assigns a new section to the most common district. It would be good to check it with the geometries of the new sections, but we don't have those.
- Working with other years. `silver_zone_metrics` would need a `year` column, and the join by sections would have to be repeated for each year, because sections change and there is only one relation file. We don't know yet how many sections would be left without an exact match, so here it is only 2023.
- BQ3: calculate the floating population only from the overnight stays (people sleeping in a zone against people living in it) and check that the results make sense.
- `silver_dim_zone` may need `municipality_id` and `gau_id` for the districts. The residence in the overnight stays is always a district, so without them we can only calculate the people living in a zone at district level. Not built here yet.